In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch07-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "07-02-maximum-common-substructure.py"

# 三個分子有幾種並列的共同片段？

第一次執行會下載 ESOL，搜尋和繪圖均不涉及隨機抽樣。小練習的程式可接在分析程式末端，再執行最後一格。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import os
import deepchem as dc
from rdkit import Chem
from rdkit.Chem import Draw, rdFMCS

# ESOL，1,128 筆水溶解度量測，log10(mol/L)，DOI 10.1021/ci034243x。
# 只取結構做比對，不切分資料，也不轉換量測值。
_, (dataset,), _ = dc.molnet.load_delaney(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter=None,
    transformers=[], reload=False, data_dir=os.environ.get("DEEPCHEM_DATA_DIR"))
structures = {}
for smiles in dataset.ids:
    mol = Chem.MolFromSmiles(smiles)
    structures[Chem.MolToSmiles(mol)] = mol

def esol_mol(smiles):
    key = Chem.MolToSmiles(Chem.MolFromSmiles(smiles))
    return structures[key]

mols = [esol_mol(s) for s in ("Oc1ccc(Cl)cc1", "COc1ccc(Cl)cc1", "COc1cccc(Cl)c1")]
names = ["4-Chlorophenol", "4-Chloroanisole", "3-Chloroanisole"]
p = rdFMCS.MCSParameters()
p.MaximizeBonds = True
p.Threshold = 1.0
p.Timeout = 10
p.StoreAll = True
p.AtomTyper = rdFMCS.AtomCompare.CompareElements
p.BondTyper = rdFMCS.BondCompare.CompareOrderExact
p.BondCompareParameters.RingMatchesRingOnly = True
p.BondCompareParameters.CompleteRingsOnly = True

def common(group):
    result = rdFMCS.FindMCS(group, p)
    if result.canceled:
        raise RuntimeError("搜尋逾時；目前的解尚未證實為最大。")
    queries = result.degenerateSmartsQueryMolDict
    print(f"{len(group)} mols: {result.numAtoms} atoms, {result.numBonds} bonds")
    for smarts in sorted(queries):
        print(smarts)
    return list(queries.values())

pair = common(mols[:2])
trio = common(mols)
# 讓三個苯環與相連的氧朝向一致，方便對照氯的位置。
from rdkit.Chem import rdDepictor
template = Chem.MolFromSmiles("Oc1ccccc1")
rdDepictor.Compute2DCoords(template)
for mol in mols:
    rdDepictor.GenerateDepictionMatching2DStructure(mol, template)

panels, atom_lists, bond_lists, legends = [], [], [], []
for label, probe in [("A", "Oc1ccccc1"), ("B", "Clc1ccccc1")]:
    query = next(q for q in trio if Chem.MolFromSmiles(probe).HasSubstructMatch(q))
    for mol, name in zip(mols, names):
        match = mol.GetSubstructMatch(query)
        assert len(match) == query.GetNumAtoms()
        bonds = [mol.GetBondBetweenAtoms(match[b.GetBeginAtomIdx()],
                                         match[b.GetEndAtomIdx()]).GetIdx()
                 for b in query.GetBonds()]
        panels.append(mol)
        atom_lists.append(match)
        bond_lists.append(bonds)
        legends.append(f"{label}   {name}")
options = Draw.MolDrawOptions()
options.setHighlightColour((0.55, 0.84, 0.83))
options.legendFontSize = 19
image = Draw.MolsToGridImage(panels, molsPerRow=3, subImgSize=(300, 200),
                             legends=legends, highlightAtomLists=atom_lists,
                             highlightBondLists=bond_lists, drawOptions=options)
out = Path("../../figures") if Path("../../_quarto.yml").is_file() else Path("figures")
out.mkdir(exist_ok=True)
image.save(out / "07-02-maximum-common-substructure.png")


In [ ]:
from IPython.display import Image, display

subprocess.run([python, str(script)], check=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "07-02-maximum-common-substructure.png")))